In [1]:
import boto3
from datetime import datetime, timezone, timedelta

# Crear cliente S3
s3 = boto3.client('s3')

# Configuración
BUCKET_NAME = "tu-bucket-aqui"
DIAS_LIMITE = 30

In [2]:
def limpiar_bucket():
    try:
        # Fecha actual y fecha límite
        ahora = datetime.now(timezone.utc)
        fecha_limite = ahora - timedelta(days=DIAS_LIMITE)

        continuation_token = None

        while True:
            # Manejar paginación
            if continuation_token:
                response = s3.list_objects_v2(
                    Bucket=BUCKET_NAME,
                    ContinuationToken=continuation_token
                )
            else:
                response = s3.list_objects_v2(Bucket=BUCKET_NAME)

            # Verificar si hay archivos
            if 'Contents' not in response:
                print("El bucket está vacío.")
                break

            for obj in response['Contents']:
                key = obj['Key']
                fecha_mod = obj['LastModified']

                # Verificar si el archivo es antiguo
                if fecha_mod < fecha_limite:
                    print(f"Eliminando archivo: {key}")
                    s3.delete_object(Bucket=BUCKET_NAME, Key=key)

            # Verificar si hay más páginas
            if response.get('IsTruncated'):
                continuation_token = response.get('NextContinuationToken')
            else:
                break

        print("Limpieza completada.")

    except Exception as e:
        print("Error:", e)


In [3]:
if __name__ == "__main__":
    limpiar_bucket()

Error: An error occurred (ExpiredToken) when calling the ListObjectsV2 operation: The provided token has expired.
